# 03 - Model Training
Train baseline models (log-transformed target), pick the best performer, then hyperparameter-tune only that model.

In [1]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load processed data + preprocessor

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

preprocessor = joblib.load('../models/preprocessor.pkl')

X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

X_train_t.shape, X_test_t.shape

((13694, 115), (3424, 115))

## 2. Log-transform the target

Car prices are right-skewed, which is part of why cheap cars were getting wildly over-predicted. Every model below is wrapped in `TransformedTargetRegressor` so it's actually trained on `log1p(price)` internally, but `.predict()` still returns real dollar values (it applies `expm1` automatically) - so nothing downstream changes.

In [3]:
def make_model(estimator):
    """Wrap any regressor so it trains on log1p(price) but predicts real dollars."""
    return TransformedTargetRegressor(
        regressor=estimator,
        func=np.log1p,
        inverse_func=np.expm1
    )

## 3. Baseline comparison (default hyperparameters)

Quick pass with untuned models to see which model family is worth investing tuning time in.

In [4]:
baseline_models = {
    'LinearRegression': make_model(LinearRegression()),
    'Ridge': make_model(Ridge(random_state=42)),
    'Lasso': make_model(Lasso(random_state=42, max_iter=5000)),
    'DecisionTree': make_model(DecisionTreeRegressor(random_state=42)),
    'RandomForest': make_model(RandomForestRegressor(random_state=42, n_jobs=-1)),
}

baseline_results = []
for name, model in baseline_models.items():
    model.fit(X_train_t, y_train)
    preds = model.predict(X_test_t)
    baseline_results.append({
        'Model': name,
        'R2': r2_score(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds))
    })

baseline_df = pd.DataFrame(baseline_results).sort_values('R2', ascending=False).reset_index(drop=True)
baseline_df

,Model,R2,MAE,RMSE
0,RandomForest,0.726057,4046.910967,7644.887914
1,DecisionTree,0.558447,5149.543127,9705.820617
2,Ridge,0.225586,8520.533131,12853.682360
3,LinearRegression,0.224320,8530.354587,12864.181196
4,Lasso,-0.152719,10633.871039,15682.041223


In [5]:
best_model_name = baseline_df.iloc[0]['Model']
print(f"Best baseline model: {best_model_name} (R2 = {baseline_df.iloc[0]['R2']:.4f})")
print("Tuning will focus only on this model.")

Best baseline model: RandomForest (R2 = 0.7261)
Tuning will focus only on this model.


## 4. Hyperparameter tuning - winner only

Wide `RandomizedSearchCV` search space, but only run for whichever model won the baseline comparison above - no time spent tuning the others.

In [ ]:
# Wide param grids, keyed by model name.
# Note the 'regressor__' prefix - required because every model is wrapped in
# TransformedTargetRegressor.
#
# max_depth deliberately does NOT include None here: unbounded trees grow huge
# and bloat the saved .pkl (a RandomForest with n_estimators=800 and no depth
# cap can easily exceed GitHub's 100MB file limit). Capping depth keeps the
# model small with little to no accuracy cost.
param_distributions = {
    'Ridge': {
        'regressor__alpha': uniform(0.01, 300)
    },
    'Lasso': {
        'regressor__alpha': uniform(0.001, 30)
    },
    'DecisionTree': {
        'regressor__max_depth': [5, 7, 10, 12, 15, 18, 20],
        'regressor__min_samples_split': randint(2, 25),
        'regressor__min_samples_leaf': randint(1, 12),
        'regressor__max_features': ['sqrt', 'log2', None]
    },
    'RandomForest': {
        'regressor__n_estimators': randint(100, 350),
        'regressor__max_depth': [8, 10, 12, 15, 18, 20],
        'regressor__min_samples_split': randint(2, 20),
        'regressor__min_samples_leaf': randint(2, 10),
        'regressor__max_features': ['sqrt', 'log2', 0.5, 0.7],
        'regressor__bootstrap': [True, False]
    }
}

estimator_lookup = {
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42, max_iter=5000),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1)
}

In [7]:
def tune_model(model, param_distributions, X_train, y_train, cv=5, scoring='r2', n_iter=40, random_state=42):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        random_state=random_state,
        verbose=1
    )
    random_search.fit(X_train, y_train)
    print(f"Best Params: {random_search.best_params_}")
    print(f"Best CV Score: {random_search.best_score_:.4f}")
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

In [8]:
if best_model_name == 'LinearRegression':
    # No hyperparameters worth tuning on plain LinearRegression - just keep the baseline fit.
    final_model = baseline_models['LinearRegression']
    best_params = 'N/A'
    best_cv_score = None
    print("LinearRegression won the baseline - nothing to tune, using it as-is.")
else:
    wrapped_estimator = make_model(estimator_lookup[best_model_name])
    final_model, best_params, best_cv_score = tune_model(
        wrapped_estimator,
        param_distributions[best_model_name],
        X_train_t, y_train,
        n_iter=40
    )

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Params: {'regressor__bootstrap': False, 'regressor__max_depth': 20, 'regressor__max_features': 0.5, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 326}
Best CV Score: 0.7372


## 5. Save best hyperparameters

In [9]:
with open('../reports/best_params.json', 'w') as f:
    json.dump({
        'best_model': best_model_name,
        'best_params': str(best_params),
        'best_cv_r2': best_cv_score
    }, f, indent=2)

best_model_name, best_params, best_cv_score

('RandomForest',
 {'regressor__bootstrap': False,
  'regressor__max_depth': 20,
  'regressor__max_features': 0.5,
  'regressor__min_samples_leaf': 4,
  'regressor__min_samples_split': 2,
  'regressor__n_estimators': 326},
 np.float64(0.7372487414882849))

## 6. Evaluate tuned model on test set

In [10]:
preds = final_model.predict(X_test_t)

r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)

final_results_df = pd.DataFrame([{
    'Model': f'{best_model_name} (tuned)',
    'R2': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse
}])

print(baseline_df)
print()
print('Best Tuned on test set:')
final_results_df

              Model        R2           MAE          RMSE
0      RandomForest  0.726057   4046.910967   7644.887914
1      DecisionTree  0.558447   5149.543127   9705.820617
2             Ridge  0.225586   8520.533131  12853.682360
3  LinearRegression  0.224320   8530.354587  12864.181196
4             Lasso -0.152719  10633.871039  15682.041223

Best Tuned on test set:


,Model,R2,MAE,MSE,RMSE
0,RandomForest (tuned),0.716806,4168.84232,6.041789e+07,7772.894361


In [11]:
final_results_df.to_csv('../reports/metrics.csv', index=False)

## 7. Save the final model

In [12]:
joblib.dump(final_model, '../models/model.pkl', compress=3)
print(f'Saved {best_model_name} (tuned) to ../models/model.pkl')

import os
size_mb = os.path.getsize('../models/model.pkl') / (1024 * 1024)
print(f'model.pkl size: {size_mb:.1f} MB')
if size_mb > 90:
    print('Warning: still close to/over GitHub\'s 100MB limit - consider Git LFS, \n'
          '         or shrink n_estimators/max_depth further in the search space above.')

Saved RandomForest (tuned) to ../models/model.pkl
model.pkl size: 32.4 MB


## 8. Summary

- Trained baseline versions of Linear Regression, Ridge, Lasso, Decision Tree, Random Forest - each wrapped in `TransformedTargetRegressor` (log1p/expm1) to correct for the right-skewed price target
- Compared baselines on the test set and picked the best-performing model family
- Ran a wide `RandomizedSearchCV` (5-fold CV, 40 iterations) hyperparameter search on **only** the winning model
- Best hyperparameters saved to `reports/best_params.json`
- Tuned model's test-set metrics saved to `reports/metrics.csv`
- Final tuned model saved to `models/model.pkl` via joblib

Next notebook: deeper evaluation (`src/evaluate.py`) - actual vs predicted plots, feature importance.